<h1 align="center"><strong>Context-Enhances Latent Diffusion for Music Spectrogram Inpainting</strong></h1>

---
The project focuses on the task of music inpainting, where a model is given a music clip with a missing
segment and is asked to reconstruct the missing portion using the surrounding musical context. We plan to
represent the audio as mel spectrogram and study the problem in the context of classical piano music, using the [MAESTRO dataset](https://magenta.withgoogle.com/datasets/maestro) due to its high quality and relative tractability

**Motivation:**

While latent diffusion models have proven to be strong in generative audio tasks, reconstruction quality can
still degrade as the missing gap becomes longer or more structurally ambiguous. In particular, a standard latent
diffusion model may not make sufficiently effective use of the left and right surrounding musical context when
restoring the missing region. Our project would therefore investigate whether stro

In [5]:
# import packages
# .py scripts imports
from utils.data_load import DataModule # used to load dataset and create dataloaders
from utils.mel2wav import Mel2Waveform # used to convert mel spectrograms back to waveforms for listening and evaluation

In [2]:
# load in dataset and create dataloaders using data_load.datamodule class
# using datamodule class to load data
dm = DataModule(
    repo_id="han2o/grant-ortsaem-processedV2", # hugging face repo id for dataset
    gap="0.5",  #  gap version [0.5, 2.0, 3.0, 5.0]
    input_key="masked_spectrogram",
    target_key="spectrogram",
    mask_key="mask",
    batch_size=8, # batch size for dataloaders
    num_workers=0,
    streaming=True,
    # maximum number of training/ validation/ test samples. Set to None to use the entire dataset.
    max_train_samples=128, 
    max_val_samples=32,
    max_test_samples=32, 
)

train_loader, val_loader, test_loader = dm.setup()

Resolving data files:   0%|          | 0/935 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/100 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/100 [00:00<?, ?it/s]

In [4]:
import torch
from cnn_model import PolimiCNNAutoencoder
from train_utils import train_one_epoch, validate
from checkpointing import load_checkpoint, save_checkpoint

# Initialize everything
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PolimiCNNAutoencoder().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# Resume if possible
start_epoch, history = load_checkpoint("phase3_cnn.pt", model, optimizer)

# Start training
for epoch in range(start_epoch, 2):
    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    val_loss = validate(model, val_loader, device)
    
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
    save_checkpoint(model, optimizer, epoch, history, "phase3_cnn.pt")

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
Training: 0it [01:32, ?it/s]


KeyboardInterrupt: 